# FOI – 230416001 – Statistics on first disposal of a C100 Application

## 1. Import packages and set options 
<a name="import_packages"></a>

In [ ]:
import pandas as pd  # a module which provides the data structures and functions to store and manipulate tables in dataframes
import pydbtools as pydb  # A module which allows SQL queries to be run on the Analytical Platform from Python, see https://github.com/moj-analytical-services/pydbtools
import boto3  # allows you to directly create, update, and delete AWS resources from Python scripts
import numpy as np
import re

#import the pacakges required to produce an excel spreadsheet
import xlsxwriter
import boto3, io, os, pathlib

# sets parameters to view dataframes for tables easier
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 900)
pd.set_option("display.max_colwidth", 200)

## 2. Define key variables to be used throughout the notebook 
<a name="define_key_variables"></a>

In [ ]:
#this is the database we will be extracting from
database = "familyman_dev_v3"
derived = "familyman_derived_dev_v3"

#this extracts the February snapshot from athena
snapshot_date = "2023-02-09"

#this is the athena database we will be storing our tables in
fcsq_database = "fcsq"

#this is the s3 bucket we will be saving data to
s3 = boto3.resource("s3")
bucket = s3.Bucket("alpha-family-data")

#setting current year and quarter
pub_quarter = 4
pub_year = 2022

## Extract date of issue

In [ ]:
pydb.create_temp_table(
f""" 
SELECT 
  case_number,
  CAST(value AS Date) AS case_DOI
FROM 
  {database}.case_fields AS f
 
WHERE
  field_model = 'FM2C_DI'
  AND mojap_snapshot_date = DATE'{snapshot_date}'

""",

"DOI")

In [ ]:
a = pydb.read_sql_query("SELECT * from __temp__.DOI")
a

## Create start date using date of issue

In [ ]:
#Adding the start date to the main child apps data created in the main child apps process (so data is at child and order level)
#Only including specific order types
pydb.create_temp_table(
f""" 
SELECT 
  t1.case_number,
  t1.receipt_date,
  t1.event,
  t1.field_model,
  t1.order_type,
  t1.order_code,
  t1.order_desc,
  t1.child_role_id,
  t1.order_case_type,
  t2.case_DOI,
  CASE WHEN  t2.case_DOI is null
        THEN t1.receipt_date
       WHEN t1.field_model = 'U22_AT' AND (t2.Case_DOI<t1.Receipt_date)
        THEN t2.Case_DOI
        ELSE t1.Receipt_date END
      AS Start_date
FROM 
  {fcsq_database}.ca_apps_child AS t1
      LEFT JOIN __temp__.DOI AS t2 
       ON t1.case_number = t2.case_number
WHERE
  t1.order_code IN (29,30,31,32)

""",

"ca_apps_issue_date")

In [ ]:
a = pydb.read_sql_query("SELECT * from __temp__.ca_apps_issue_date")
a

## Matching applications to disposals

In [ ]:
#Matching apps data to the main child disposal data set created in the child disposals processing - matches at child and order level
#Filetering out certain order types alongside interim orders
#Ranks the earlies diposal as the first disposal
pydb.create_temp_table(
f""" 
SELECT 
  a.*,
  YEAR(d.disp_date) AS Year,
  QUARTER(d.disp_date) AS Quarter,
  MONTH(d.disp_date) AS Month,
  d.disp_date,
  date_diff('day',a.start_date, d.disp_date) AS days,
  date_diff('week',a.start_date, d.disp_date) AS weeks,
  ROW_NUMBER() OVER(PARTITION BY a.case_number, a.child_role_id, a.order_code, a.start_date
                       ORDER BY d.disp_date) 
      AS disp_rank,
  d.disposal_court,
  d.disposal_dfj,
  d.disposal_region 
FROM
  __temp__.ca_apps_issue_date a
  LEFT JOIN {fcsq_database}.ca_disps_all_children d
    ON a.case_number = d.case_number
    AND a.order_case_type = d.order_case_type
    AND a.child_role_id = d.child_role_id
WHERE 
  date_diff('day',a.start_date, d.disp_date) >= 0
  AND d.event_model NOT IN ('CPA','C21','C27','C30','C31','C33','C35B','C44A','C44B','C46A','C46B','C47A','C47C','C48A', 
                          'C48B','C48C','C49','D51','D84C','MAGEPO','MAGS37')
  AND d.disp_type_code <> 2

""",

"app_disp_match")

In [ ]:
a = pydb.read_sql_query("SELECT * from __temp__.app_disp_match")
a

## Selecting the earliest disposal

In [ ]:
#Selects the first diposal for each order type/child matched
#Multiple children can get matched to the same disposal (if the disposal has multiple child id's)
#More than one application/order type can be matched to one disposal - here we are replicating the SAS process 
pydb.create_temp_table(
f""" 
SELECT 
  *
FROM 
  __temp__.app_disp_match
WHERE
  disp_rank = 1

""",

"first_disp")

In [ ]:
a = pydb.read_sql_query("SELECT * from __temp__.first_disp")
a

## Creation of main timeliess dataset, deletion of dup apps

In [ ]:
#Here we get rid of any instances of an application of the same order type for the same child, and only the earliest one is selected
pydb.create_temp_table(
f""" 
WITH dup_app_type AS
 (SELECT 
   *,
   ROW_NUMBER() OVER(PARTITION BY case_number, child_role_id, order_code
                       ORDER BY start_date, case_number, child_role_id, order_code) 
     AS dup_app_rank
 FROM 
   __temp__.first_disp
 )

SELECT
  *,
  cast((EVENT / 100000000) as int)  AS EVENT_COURT
FROM
  dup_app_type
WHERE
  dup_app_rank = 1

""",

"ca_timeliness_all")

In [ ]:
a = pydb.read_sql_query("SELECT * from __temp__.ca_timeliness_all limit 10")
a

## Adding harm flags to c100 applications

In [ ]:
#Adding the harm alleged column to the table from the derived 'cases_derived' table to help filter out c100 applications that have or don't have harm flags
pydb.create_temp_table(
f""" 

SELECT DISTINCT t1.*,
t2.harm_alleged
from __temp__.ca_timeliness_all t1
LEFT JOIN {derived}.cases_derived t2
ON t1.CASE_NUMBER = t2.CASE_NUMBER 


""",

"c100_apps_harm_flag")

In [ ]:
a = pydb.read_sql_query("SELECT * from __temp__.c100_apps_harm_flag limit 10")
a

## Table 1i

In [ ]:
#Creates a table with the number of private law c100 applications by year and the average time taken to reach final disposal.
#Filters the data to look for 2014 < year < 2023 and private law order_case_type 'P'

pydb.create_temp_table(
f""" 
 SELECT
  'Children Act (Private)' as Case_type,
  Year,
  COUNT(*) AS count,
  ROUND(AVG ((days)/7),1) as mean_duration
FROM
  __temp__.ca_timeliness_all
WHERE
  order_case_type = 'P' 
  AND YEAR > 2014 
  AND YEAR < 2023
GROUP BY
  year
""",

"table_1i")

In [ ]:
table_1i = pydb.read_sql_query("SELECT * from __temp__.table_1i order by year")
table_1i

## Table 1ii

In [ ]:
#Creates a table with the number of private law c100 applications by year and disposal_region, with the average time taken to reach final disposal.
#Filters the data to look for 2014 < year < 2023 and private law order_case_type 'P'

pydb.create_temp_table(
f""" 
 SELECT
  'Children Act (Private)' as Case_type,
  Year,
  disposal_region AS region,
  COUNT(*) AS count,
  ROUND(AVG ((days)/7),1) as mean_duration
FROM
  __temp__.ca_timeliness_all
WHERE
  order_case_type = 'P' 
  AND YEAR > 2014 
  AND YEAR < 2023
GROUP BY
  year,
  disposal_region
""",

"table_1ii")

In [ ]:
table_1ii = pydb.read_sql_query("SELECT * from __temp__.table_1ii order by year, region")
table_1ii

## Table 1iii

In [ ]:
#Creates a table with the number of private law c100 applications by year and disposal_region, with the average time taken to reach final disposal.
#Filters the data to look for 2014 < year < 2023, private law order_case_type 'P', and disposal_court in ('SOUTHEND', 'BURY ST. EDMUNDS', 'COLCHESTER', 'CHELMSFORD', 'IPSWICH')

pydb.create_temp_table(
f""" 
 SELECT 
  'Children Act (Private)' as Case_type,
  Year,
  'Essex & Suffolk DFJ' AS dfj, 
  disposal_region AS region,
  COUNT(*) AS count,
  ROUND(AVG ((days)/7),1) as mean_duration
FROM
  __temp__.ca_timeliness_all
WHERE
  order_case_type = 'P' 
  AND YEAR > 2014 
  AND YEAR < 2023
  AND disposal_court in ('SOUTHEND', 'BURY ST. EDMUNDS', 'COLCHESTER', 'CHELMSFORD', 'IPSWICH')
GROUP BY
  year,
  disposal_region
ORDER BY
  year
""",

"table_1iii")

In [ ]:
table_1iii = pydb.read_sql_query("SELECT * from __temp__.table_1iii")
table_1iii

## Table 2i

In [ ]:
#Creates a table with the number of private law c100 applications (with harm alleged) by year, with the average time taken to reach final disposal.
#Filters the data to look for 2014 < year < 2023, private law order_case_type 'P', and harm alleged 'Y'

pydb.create_temp_table(
f""" 
 SELECT
  'Children Act (Private)' as Case_type,
  Year,
  COUNT(*) AS count,
  ROUND(AVG ((days)/7),1) as mean_duration
FROM
  __temp__.c100_apps_harm_flag
WHERE
  order_case_type = 'P' 
  AND harm_alleged = 'Y'
  AND YEAR > 2014 
  AND YEAR < 2023
GROUP BY
  year
""",

"table_2i")

In [ ]:
table_2i = pydb.read_sql_query("SELECT * from __temp__.table_2i order by year")
table_2i

## Table 2ii

In [ ]:
#Creates a table with the number of private law c100 applications (with harm alleged) by year and disposal_region, with the average time taken to reach final disposal.
#Filters the data to look for 2014 < year < 2023, private law order_case_type 'P', and harm alleged 'Y'

pydb.create_temp_table(
f""" 
 SELECT
  'Children Act (Private)' as Case_type,
  Year,
  disposal_region AS region,
  COUNT(*) AS count,
  ROUND(AVG ((days)/7),1) as mean_duration
FROM
  __temp__.c100_apps_harm_flag
WHERE
  order_case_type = 'P'
  AND harm_alleged = 'Y'
  AND YEAR > 2014 
  AND YEAR < 2023
GROUP BY
  year,
  disposal_region
""",

"table_2ii")

In [ ]:
table_2ii = pydb.read_sql_query("SELECT * from __temp__.table_2ii order by year, region")
table_2ii

## Table 2iii

In [ ]:
#Creates a table with the number of private law c100 applications (with harm alleged) by year and disposal_region, with the average time taken to reach final disposal.
#Filters the data to look for 2014 < year < 2023, private law order_case_type 'P', harm alleged 'Y', and disposal_court in ('SOUTHEND', 'BURY ST. EDMUNDS', 'COLCHESTER', 'CHELMSFORD', 'IPSWICH')

pydb.create_temp_table(
f""" 
 SELECT 
  'Children Act (Private)' as Case_type,
  Year,
  'Essex & Suffolk DFJ' AS dfj, 
  disposal_region AS region,
  COUNT(*) AS count,
  ROUND(AVG ((days)/7),1) as mean_duration
FROM
  __temp__.c100_apps_harm_flag
WHERE
  order_case_type = 'P' 
  AND harm_alleged = 'Y'
  AND YEAR > 2014 
  AND YEAR < 2023
  AND disposal_court in ('SOUTHEND', 'BURY ST. EDMUNDS', 'COLCHESTER', 'CHELMSFORD', 'IPSWICH')
GROUP BY
  year,
  disposal_region
ORDER BY
  year
""",

"table_2iii")

In [ ]:
table_2iii = pydb.read_sql_query("SELECT * from __temp__.table_2iii")
table_2iii

## Create Final Output

In [ ]:
#creating a panda dataframe for all 4 tables (1A, 1B, 2A, 2B)
#uses the queries underneath table 1i, 1ii, 1iii, 2i, 2ii, and 2iii
df1 = pd.DataFrame(table_1i)

df2 = pd.DataFrame(table_1ii)

df3 = pd.DataFrame(table_1iii)

df4 = pd.DataFrame(table_2i)

df5 = pd.DataFrame(table_2ii)

df6 = pd.DataFrame(table_2iii)

In [ ]:
#creates a seperate sheet for each dataframe 
with io.BytesIO() as xlsx_data:
    with pd.ExcelWriter(xlsx_data) as writer:
        
        df1.to_excel(writer, sheet_name='T1a', index=False)

        df2.to_excel(writer, sheet_name='T1b', index=False)

        df3.to_excel(writer, sheet_name='T1c', index=False)

        df4.to_excel(writer, sheet_name='T2a', index=False)
        
        df5.to_excel(writer, sheet_name='T2b', index=False)
        
        df6.to_excel(writer, sheet_name='T2c', index=False)
    ## Upload the in-memory data to S3
    s3 = boto3.resource("s3")
    bucket = s3.Bucket("alpha-family-data")
    object_name = "FOIs/FOI 230416001 – working table.xlsx"
    s3.Bucket("alpha-family-data").put_object(Key=object_name, Body=xlsx_data.getvalue())